# Q-IFU: A JWST/NIRSpec Integral Field Unit Survey of Luminous Quasars at z ∼ 5–6: Rest-frame Optical Nuclear Properties and Extended Nebulae

Paper: https://iopscience.iop.org/article/10.3847/1538-4357/ae99e0

# Part 1. Powerful Quasar Outflows in the Early Universe


In [ ]:
# ============================================================
# QUASAR OUTFLOW → ENHANCED [O III]
# Inspired by Liu et al. (2026), ApJ 1009:176
#
# One self-contained Jupyter cell.
# Requires: numpy, matplotlib, ffmpeg
#
# Physical geometry and spectral evolution are SCHEMATIC.
# This is not a hydrodynamic simulation and the spectrum
# is not digitized from the paper.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Polygon
from IPython.display import Video, display

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT = "mp4"       # "webm" or "mp4"

FPS = 30
DURATION = 18
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "ae99e0_quasar_outflow_oiii.webm"
else:
    OUTFILE = "ae99e0_quasar_outflow_oiii.mp4"


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


def gaussian(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2)


# ------------------------------------------------------------
# RANDOM SEED
# ------------------------------------------------------------

rng = np.random.default_rng(42)


# ------------------------------------------------------------
# FIGURE
# ------------------------------------------------------------

fig = plt.figure(figsize=(12.8, 7.2), facecolor="#02050b")

ax = fig.add_axes([0.035, 0.13, 0.57, 0.75])
sp = fig.add_axes([0.645, 0.18, 0.325, 0.62])

ax.set_facecolor("#02050b")
sp.set_facecolor("#02050b")

ax.set_xlim(-1.35, 1.35)
ax.set_ylim(-1.02, 1.02)
ax.set_aspect("equal")
ax.axis("off")


# ------------------------------------------------------------
# TITLES
# ------------------------------------------------------------

fig.text(
    0.5, 0.955,
    "QUASAR OUTFLOW → ENHANCED [O III]",
    ha="center", va="center",
    color="white",
    fontsize=20,
    fontweight="bold"
)

fig.text(
    0.5, 0.918,
    "Schematic view of fast gas outflows in a z ≈ 5–6 quasar",
    ha="center", va="center",
    color="#B8D4E8",
    fontsize=12
)


# ------------------------------------------------------------
# BACKGROUND STARS
# ------------------------------------------------------------

n_bg = 180

sx = rng.uniform(-1.35, 1.35, n_bg)
sy = rng.uniform(-1.02, 1.02, n_bg)

ss = rng.uniform(1.0, 7.0, n_bg)
sa = rng.uniform(0.10, 0.48, n_bg)

star_colors = np.ones((n_bg, 4))
star_colors[:, :3] = [0.72, 0.86, 1.0]
star_colors[:, 3] = sa

ax.scatter(
    sx, sy,
    s=ss,
    c=star_colors,
    linewidths=0,
    zorder=0
)


# ------------------------------------------------------------
# HOST GALAXY
# ------------------------------------------------------------

n_host = 750

theta = rng.uniform(0, 2*np.pi, n_host)

r = rng.exponential(0.32, n_host)
r = np.clip(r, 0, 1.0)

gx = 1.08 * r * np.cos(theta)
gy = 0.48 * r * np.sin(theta)

host_alpha = (
    0.05
    + 0.23 * np.exp(-2.3 * r)
)

host_colors = np.zeros((n_host, 4))
host_colors[:, 0] = 0.30
host_colors[:, 1] = 0.56
host_colors[:, 2] = 0.72
host_colors[:, 3] = host_alpha

ax.scatter(
    gx, gy,
    s=rng.uniform(3, 18, n_host),
    c=host_colors,
    linewidths=0,
    zorder=1
)


# ------------------------------------------------------------
# GAS CLOUDS
# ------------------------------------------------------------

N_GAS = 180

gas_angle = rng.uniform(0, 2*np.pi, N_GAS)
gas_radius = rng.uniform(0.20, 0.95, N_GAS)

gas_x0 = gas_radius * np.cos(gas_angle)
gas_y0 = 0.58 * gas_radius * np.sin(gas_angle)

gas_sizes = rng.uniform(12, 38, N_GAS)

gas_colors = np.zeros((N_GAS, 4))
gas_colors[:, :3] = [0.34, 0.60, 0.72]
gas_colors[:, 3] = rng.uniform(0.12, 0.32, N_GAS)

gas_scatter = ax.scatter(
    gas_x0,
    gas_y0,
    s=gas_sizes,
    c=gas_colors,
    linewidths=0,
    zorder=3
)


# ------------------------------------------------------------
# BLACK HOLE
# ------------------------------------------------------------

bh_glow3 = Circle(
    (0, 0), 0.105,
    facecolor="#7FDBFF",
    edgecolor="none",
    alpha=0.05,
    zorder=10
)

bh_glow2 = Circle(
    (0, 0), 0.070,
    facecolor="#9CE8FF",
    edgecolor="none",
    alpha=0.10,
    zorder=11
)

bh_glow1 = Circle(
    (0, 0), 0.040,
    facecolor="white",
    edgecolor="none",
    alpha=0.55,
    zorder=12
)

bh = Circle(
    (0, 0), 0.018,
    facecolor="#000000",
    edgecolor="#E9FBFF",
    linewidth=1.2,
    zorder=15
)

for p in [bh_glow3, bh_glow2, bh_glow1, bh]:
    ax.add_patch(p)


# ------------------------------------------------------------
# ACCRETION DISK
# ------------------------------------------------------------

disk_x = np.linspace(-0.24, 0.24, 250)

disk_y1 = 0.035 * np.sin(disk_x * 10)
disk_y2 = -disk_y1

disk_top, = ax.plot(
    disk_x,
    disk_y1,
    color="#EAFBFF",
    lw=3.2,
    alpha=0.90,
    zorder=14
)

disk_bottom, = ax.plot(
    disk_x,
    disk_y2,
    color="#64CFFF",
    lw=2.0,
    alpha=0.60,
    zorder=13
)


# ------------------------------------------------------------
# BIPOLAR OUTFLOW CONES
# ------------------------------------------------------------

upper_cone = Polygon(
    [[-0.035, 0.02],
     [0.035, 0.02],
     [0.40, 0.92],
     [-0.40, 0.92]],
    closed=True,
    facecolor="#49C9FF",
    edgecolor="#79DDFF",
    linewidth=1.0,
    alpha=0.0,
    zorder=2
)

lower_cone = Polygon(
    [[-0.035, -0.02],
     [0.035, -0.02],
     [0.40, -0.92],
     [-0.40, -0.92]],
    closed=True,
    facecolor="#49C9FF",
    edgecolor="#79DDFF",
    linewidth=1.0,
    alpha=0.0,
    zorder=2
)

ax.add_patch(upper_cone)
ax.add_patch(lower_cone)


# ------------------------------------------------------------
# OUTFLOW PARTICLES
# ------------------------------------------------------------

N_OUT = 380

out_side = rng.choice([-1, 1], N_OUT)

out_phase = rng.uniform(0, 1, N_OUT)
out_spread = rng.normal(0, 0.12, N_OUT)

out_scatter = ax.scatter(
    np.zeros(N_OUT),
    np.zeros(N_OUT),
    s=rng.uniform(5, 18, N_OUT),
    c="#77DDFF",
    alpha=0.0,
    linewidths=0,
    zorder=6
)


# ------------------------------------------------------------
# [O III] IONIZED GAS
# ------------------------------------------------------------

oiii_scatter = ax.scatter(
    gas_x0,
    gas_y0,
    s=gas_sizes * 1.8,
    facecolors="none",
    edgecolors="#A5F6FF",
    linewidths=1.2,
    alpha=0.0,
    zorder=7
)


# ------------------------------------------------------------
# LABELS — LEFT PANEL
# ------------------------------------------------------------

object_label = ax.text(
    0, -0.18,
    "ACTIVE GALACTIC NUCLEUS",
    ha="center",
    va="top",
    color="#DDF7FF",
    fontsize=11,
    fontweight="bold",
    alpha=0
)

outflow_label = ax.text(
    0.44, 0.70,
    "FAST OUTFLOW",
    ha="left",
    va="center",
    color="#80E2FF",
    fontsize=11,
    fontweight="bold",
    alpha=0
)

gas_label = ax.text(
    -0.92, 0.56,
    "IONIZED GAS",
    ha="left",
    va="center",
    color="#DDFBFF",
    fontsize=11,
    fontweight="bold",
    alpha=0
)

oiii_label = ax.text(
    -0.92, 0.48,
    "[O III] EMISSION",
    ha="left",
    va="center",
    color="#9FF5FF",
    fontsize=12,
    fontweight="bold",
    alpha=0
)


# ------------------------------------------------------------
# SPECTRUM
# ------------------------------------------------------------

wave = np.linspace(4700, 5100, 1000)

continuum = (
    0.18
    + 0.00010 * (wave - 4900)
)

hbeta_profile = gaussian(wave, 4861, 20)

oiii_4959 = gaussian(wave, 4959, 7)
oiii_5007 = gaussian(wave, 5007, 8)

base_spectrum = (
    continuum
    + 0.40 * hbeta_profile
)

sp.set_xlim(4700, 5100)
sp.set_ylim(0, 2.15)

sp.set_xlabel(
    "REST-FRAME WAVELENGTH  [Å]",
    color="#B5CDE0",
    fontsize=11,
    fontweight="bold",
    labelpad=10
)

sp.set_ylabel(
    "RELATIVE FLUX",
    color="#B5CDE0",
    fontsize=11,
    fontweight="bold",
    labelpad=8
)

sp.tick_params(
    colors="#AFC7D9",
    labelsize=10,
    length=4
)

for spine in sp.spines.values():
    spine.set_color("#52738A")
    spine.set_linewidth(1.0)

sp.grid(
    alpha=0.18,
    linewidth=0.7,
    color="#638399"
)

sp.set_title(
    "REST-FRAME OPTICAL SPECTRUM",
    color="#C2D9E9",
    fontsize=11,
    fontweight="bold",
    pad=13
)


# reference spectrum

reference_line, = sp.plot(
    wave,
    base_spectrum,
    color="#6EBBDF",
    lw=1.3,
    alpha=0.32,
    linestyle="--"
)

spectrum_line, = sp.plot(
    wave,
    base_spectrum,
    color="#DFF8FF",
    lw=2.5,
    alpha=0.95
)


# line markers

sp.axvline(
    4861,
    color="#7194AA",
    lw=1,
    alpha=0.35
)

sp.axvline(
    4959,
    color="#7194AA",
    lw=1,
    alpha=0.35
)

sp.axvline(
    5007,
    color="#7194AA",
    lw=1,
    alpha=0.35
)


hbeta_text = sp.text(
    4861, 1.78,
    "Hβ",
    ha="center",
    color="#BCD8E8",
    fontsize=11,
    fontweight="bold"
)

o3a_text = sp.text(
    4959, 1.78,
    "[O III]\n4959",
    ha="center",
    color="#A8EFFF",
    fontsize=10,
    fontweight="bold"
)

o3b_text = sp.text(
    5007, 1.78,
    "[O III]\n5007",
    ha="center",
    color="#A8EFFF",
    fontsize=10,
    fontweight="bold"
)


boost_text = sp.text(
    0.5, 0.08,
    "ENHANCED [O III]",
    transform=sp.transAxes,
    ha="center",
    va="center",
    color="white",
    fontsize=13,
    fontweight="bold",
    alpha=0
)


# ------------------------------------------------------------
# PHASE TEXT
# ------------------------------------------------------------

phase_text = fig.text(
    0.32, 0.845,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold"
)

phase_sub = fig.text(
    0.32, 0.812,
    "",
    ha="center",
    va="center",
    color="#B7D3E7",
    fontsize=11
)


# ------------------------------------------------------------
# FINAL MESSAGE
# ------------------------------------------------------------

final_box = fig.text(
    0.5, 0.49,
    "FAST OUTFLOWS\n→\nENHANCED [O III] EMISSION",
    ha="center",
    va="center",
    color="white",
    fontsize=19,
    fontweight="bold",
    bbox=dict(
        boxstyle="round,pad=0.65",
        facecolor="#071521",
        edgecolor="#80DFFF",
        linewidth=1.4,
        alpha=0.0          # IMPORTANT: initially invisible
    ),
    alpha=0.0,
    zorder=100
)


final_note = fig.text(
    0.5, 0.345,
    "Observed in luminous quasars at z ≈ 5–6",
    ha="center",
    va="center",
    color="#BDEAFF",
    fontsize=12,
    alpha=0,
    zorder=101
)


# ------------------------------------------------------------
# TIMELINE
# ------------------------------------------------------------

timeline_bg = plt.Line2D(
    [0.08, 0.92],
    [0.065, 0.065],
    transform=fig.transFigure,
    color="#385B70",
    lw=2.5,
    solid_capstyle="round"
)

timeline_progress = plt.Line2D(
    [0.08, 0.08],
    [0.065, 0.065],
    transform=fig.transFigure,
    color="#83E0FF",
    lw=3.0,
    solid_capstyle="round"
)

fig.lines.extend([
    timeline_bg,
    timeline_progress
])

# ------------------------------------------------------------
# SIMPLE TIME DIAL — lower-left corner
# Decorative animation-time indicator, not a physical timescale
# ------------------------------------------------------------

from matplotlib.patches import Circle

dial_center = (-1.08, -0.76)
dial_radius = 0.115

# outer rings
dial_outer = Circle(
    dial_center,
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.78,
    zorder=30
)
ax.add_patch(dial_outer)

dial_inner = Circle(
    dial_center,
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)
ax.add_patch(dial_inner)

# 12 simple tick marks
for i in range(12):
    a = 2 * np.pi * i / 12

    r1 = dial_radius * (0.67 if i % 3 else 0.60)
    r2 = dial_radius * 0.83

    x1 = dial_center[0] + r1 * np.sin(a)
    y1 = dial_center[1] + r1 * np.cos(a)

    x2 = dial_center[0] + r2 * np.sin(a)
    y2 = dial_center[1] + r2 * np.cos(a)

    ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.15 if i % 3 else 1.7,
        alpha=0.72,
        zorder=32
    )

# rotating hand
hand_length = dial_radius * 0.57

dial_hand, = ax.plot(
    [dial_center[0], dial_center[0]],
    [dial_center[1], dial_center[1] + hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

# center pin
dial_pin = Circle(
    dial_center,
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)
ax.add_patch(dial_pin)



# ------------------------------------------------------------
# UPDATE
# ------------------------------------------------------------

def update(frame):

    t = frame / (NFRAMES - 1)

    # --------------------------------------------------------
    # PHASES
    # --------------------------------------------------------

    if t < 0.18:
        phase_text.set_text("ACTIVE QUASAR")
        phase_sub.set_text("Accretion powers the central engine")

    elif t < 0.34:
        phase_text.set_text("QUASAR WIND")
        phase_sub.set_text("Gas is accelerated away from the nucleus")

    elif t < 0.56:
        phase_text.set_text("FAST OUTFLOW")
        phase_sub.set_text("The outflow propagates through the host galaxy")

    elif t < 0.76:
        phase_text.set_text("IONIZED GAS")
        phase_sub.set_text("Fast outflows are associated with strong [O III] emission")

    else:
        phase_text.set_text("BOOSTED [O III]")
        phase_sub.set_text("A signature associated with fast outflowing gas")


    # --------------------------------------------------------
    # CENTRAL ENGINE
    # --------------------------------------------------------

    engine = smoothstep(t / 0.16)

    object_label.set_alpha(engine)

    pulse = 0.5 + 0.5 * np.sin(frame * 0.13)

    bh_glow3.set_alpha(
        0.04 + 0.06 * engine + 0.025 * pulse
    )

    bh_glow2.set_alpha(
        0.08 + 0.10 * engine + 0.035 * pulse
    )

    bh_glow1.set_alpha(
        0.35 + 0.38 * engine
    )


    # --------------------------------------------------------
    # OUTFLOW DEVELOPMENT
    # --------------------------------------------------------

    wind = smoothstep(
        (t - 0.17) / 0.22
    )

    upper_cone.set_alpha(
        0.035 * wind
    )

    lower_cone.set_alpha(
        0.035 * wind
    )

    outflow_label.set_alpha(
        smoothstep(
            (t - 0.27) / 0.10
        )
    )


    # particles repeatedly stream outward

    flow_age = (
        out_phase + 2.0 * wind
    ) % 1.0

    max_dist = 0.92 * wind

    oy = (
        out_side
        * flow_age
        * max_dist
    )

    ox = (
        out_spread
        * flow_age
        * 2.1
    )

    out_scatter.set_offsets(
        np.column_stack([ox, oy])
    )

    out_alpha = (
        wind
        * (0.18 + 0.70 * (1.0 - flow_age))
    )

    out_colors = np.zeros((N_OUT, 4))

    out_colors[:, 0] = 0.38
    out_colors[:, 1] = 0.83
    out_colors[:, 2] = 1.0
    out_colors[:, 3] = out_alpha

    out_scatter.set_facecolors(
        out_colors
    )


    # --------------------------------------------------------
    # GAS RESPONSE
    # --------------------------------------------------------

    interaction = smoothstep(
        (t - 0.40) / 0.25
    )

    # gas becomes more visible / ionized

    current_gas_colors = gas_colors.copy()

    current_gas_colors[:, 0] = (
        0.34
        + 0.25 * interaction
    )

    current_gas_colors[:, 1] = (
        0.60
        + 0.25 * interaction
    )

    current_gas_colors[:, 2] = (
        0.72
        + 0.20 * interaction
    )

    current_gas_colors[:, 3] = (
        gas_colors[:, 3]
        + 0.32 * interaction
    )

    gas_scatter.set_facecolors(
        current_gas_colors
    )

    oiii_scatter.set_alpha(
        0.65 * interaction
    )

    gas_label.set_alpha(
        smoothstep(
            (t - 0.48) / 0.10
        )
    )

    oiii_label.set_alpha(
        smoothstep(
            (t - 0.55) / 0.10
        )
    )


    # --------------------------------------------------------
    # SPECTRUM
    # --------------------------------------------------------

    oiii_strength = (
        0.12
        + 1.18 * interaction
    )

    spectrum = (
        base_spectrum
        + oiii_strength
        * (
            0.34 * oiii_4959
            + 0.92 * oiii_5007
        )
    )

    spectrum_line.set_ydata(
        spectrum
    )

    boost_text.set_alpha(
        smoothstep(
            (t - 0.58) / 0.12
        )
    )


    # --------------------------------------------------------
    # FINAL CARD
    #
    # Appears only near the end.
    # ~13.1 s: starts fading in
    # ~13.9 s: fully visible
    # ~14–18 s: static final hold
    # --------------------------------------------------------

    final_progress = smoothstep(
        (t - 0.73) / 0.045
    )

    # text
    final_box.set_alpha(final_progress)

    # bbox must be animated separately
    final_box.get_bbox_patch().set_alpha(
        0.94 * final_progress
    )

    # explanatory line appears together with the card
    final_note_progress = smoothstep(
        (t - 0.745) / 0.04
    )

    final_note.set_alpha(final_note_progress)


    # --------------------------------------------------------
    # TIMELINE
    # --------------------------------------------------------

    timeline_progress.set_xdata(
        [0.08, 0.08 + 0.84 * t]
    )

    # --------------------------------------------------------
    # TIME DIAL
    # One clockwise revolution over the full 18 s animation.
    # This represents animation progress, not physical time.
    # --------------------------------------------------------

    angle = 2.0 * np.pi * t

    hand_x = (
        dial_center[0]
        + hand_length * np.sin(angle)
    )

    hand_y = (
        dial_center[1]
        + hand_length * np.cos(angle)
    )

    dial_hand.set_data(
        [dial_center[0], hand_x],
        [dial_center[1], hand_y]
    )


    return []


# ------------------------------------------------------------
# ANIMATION
# ------------------------------------------------------------

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ------------------------------------------------------------
# EXPORT
# ------------------------------------------------------------

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(f"Saved: {OUTFILE}")

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# Мощные ветры квазаров в ранней Вселенной

С помощью космического телескопа James Webb астрономы исследовали 27 ярких квазаров с красным смещением z ≈ 5–6. Мы видим эти объекты в эпоху, когда Вселенной было около миллиарда лет. Наблюдения JWST/NIRSpec позволили изучить оптические спектральные линии квазаров и газа в окружающих их галактиках.

Особое внимание исследователи уделили линии дважды ионизованного кислорода [O III]. У некоторых квазаров она оказалась необычно сильной. При этом её интенсивность практически не зависит от светимости квазара, массы центральной чёрной дыры или эддингтоновского отношения. Авторы связывают усиление [O III] прежде всего с быстрыми потоками газа, выбрасываемыми активным ядром галактики.

Вещество, ускоренное в окрестностях сверхмассивной чёрной дыры, распространяется через галактику и взаимодействует с окружающим газом. Именно этот быстро движущийся газ способен создавать особенно сильное излучение [O III]. Поэтому спектральные линии позволяют обнаруживать воздействие квазара на окружающую среду даже в очень молодой Вселенной.

Исследование показывает, что мощные квазарные истечения уже существовали при z ≈ 5–6 и могли существенно воздействовать на газ галактик, в которых находились эти сверхмассивные чёрные дыры.

Анимация схематическая. Геометрия истечения и изменение спектра используются для иллюстрации физической идеи исследования и не являются прямой визуализацией измеренных данных.

# Powerful Quasar Outflows in the Early Universe

Using the James Webb Space Telescope, astronomers studied 27 luminous quasars at redshifts of z ≈ 5–6. We observe these objects as they existed when the Universe was roughly a billion years old. JWST/NIRSpec observations allowed the researchers to investigate rest-frame optical emission lines from the quasars and gas in their surrounding galaxies.

Particular attention was given to emission from doubly ionized oxygen, [O III]. In several quasars this emission is unusually strong. Its strength shows very little dependence on quasar luminosity, central black-hole mass, or Eddington ratio. Instead, the authors conclude that the enhanced [O III] emission is primarily associated with fast gas outflows from the active galactic nucleus.

Material accelerated near the supermassive black hole can propagate through the host galaxy and interact with its surrounding gas. Emission from this rapidly moving material can produce particularly strong [O III] signatures. These spectral lines therefore provide a way to trace the influence of quasars on their environments even in the young Universe.

The observations show that powerful quasar outflows were already present at z ≈ 5–6 and could interact strongly with gas in the galaxies hosting these rapidly growing supermassive black holes.

The animation is schematic. The geometry of the outflow and the evolving spectrum illustrate the physical interpretation of the study rather than directly reproducing the measured data.

---

# Part 2. How JWST Reveals Gas Around Distant Quasars

In [ ]:
# ============================================================
# JWST/NIRSpec IFU — REVEALING EXTENDED [O III]
# Inspired by Liu et al. (2026), ApJ 1009:176
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# TIMING
#   0.0–13.2 s  Observation → PSF model → subtraction
#   13.2–14.0 s Transition
#   14.0–20.0 s Final result screen
#
# IMPORTANT
# Spatial morphology and maps are SCHEMATIC.
# This is not a reconstruction of a specific Q-IFU object.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "mp4"       # "webm" or "mp4"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "ae99e0_jwst_ifu_extended_oiii.webm"
else:
    OUTFILE = "ae99e0_jwst_ifu_extended_oiii.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


def gauss2d(x, y, x0, y0, sx, sy, amp=1.0):
    return amp * np.exp(
        -0.5 * (
            ((x - x0) / sx) ** 2 +
            ((y - y0) / sy) ** 2
        )
    )


rng = np.random.default_rng(42)


# ============================================================
# FIGURE
# ============================================================

BG = "#02050b"

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# GLOBAL TITLE
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "JWST/NIRSpec IFU: REVEALING EXTENDED [O III]",
    ha="center",
    va="center",
    color="white",
    fontsize=20,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.918,
    "Separating a luminous quasar from faint surrounding gas",
    ha="center",
    va="center",
    color="#B8D4E8",
    fontsize=12,
    zorder=200
)


# ============================================================
# SYNTHETIC DATA
# ============================================================

N = 31

coord = np.linspace(
    -1.5,
    1.5,
    N
)

X, Y = np.meshgrid(
    coord,
    coord
)

R2 = X**2 + Y**2


# ------------------------------------------------------------
# CENTRAL QUASAR PSF
# ------------------------------------------------------------

psf_core = (
    7.5 *
    np.exp(
        -R2 /
        (2 * 0.115**2)
    )
)

psf_mid = (
    2.4 *
    np.exp(
        -R2 /
        (2 * 0.26**2)
    )
)

psf_wing = (
    0.75 *
    np.exp(
        -R2 /
        (2 * 0.52**2)
    )
)

PSF = (
    psf_core +
    psf_mid +
    psf_wing
)


# ------------------------------------------------------------
# EXTENDED [O III]
# Schematic asymmetric / clumpy nebula
# ------------------------------------------------------------

nebula = np.zeros_like(X)

nebula += gauss2d(
    X, Y,
    0.38, 0.15,
    0.34, 0.20,
    1.00
)

nebula += gauss2d(
    X, Y,
    0.76, 0.42,
    0.27, 0.19,
    0.72
)

nebula += gauss2d(
    X, Y,
    -0.43, -0.27,
    0.25, 0.36,
    0.60
)

nebula += gauss2d(
    X, Y,
    -0.76, 0.18,
    0.20, 0.17,
    0.34
)

nebula += gauss2d(
    X, Y,
    0.12, -0.71,
    0.18, 0.28,
    0.48
)

noise_field = rng.normal(
    0,
    1,
    (N, N)
)

nebula *= (
    1.0 +
    0.13 * noise_field
)

nebula = np.clip(
    nebula,
    0,
    None
)

nebula /= nebula.max()


# ------------------------------------------------------------
# OBSERVED FIELD
# ------------------------------------------------------------

observed = (
    PSF +
    1.25 * nebula
)

observed /= observed.max()


# ------------------------------------------------------------
# SCHEMATIC VELOCITY FIELD
# ------------------------------------------------------------

velocity = (
    210 *
    np.tanh(
        X / 0.55
    )
    +
    45 *
    np.sin(
        2.4 * Y
    )
)

velocity *= (
    0.72 +
    0.28 * nebula
)

velocity = np.clip(
    velocity,
    -300,
    300
)


# ------------------------------------------------------------
# SCHEMATIC VELOCITY DISPERSION
# ------------------------------------------------------------

dispersion = (
    210
    +
    230 * nebula
    +
    35 *
    np.exp(
        -R2 / 0.30
    )
)

dispersion = np.clip(
    dispersion,
    200,
    500
)


# ============================================================
# SCENE 1 — MAIN IFU FIELD
# ============================================================

ax = fig.add_axes(
    [0.045, 0.13, 0.55, 0.72],
    zorder=10
)

ax.set_facecolor(BG)

ax.set_xlim(
    -1.55,
    1.55
)

ax.set_ylim(
    -1.55,
    1.55
)

ax.set_aspect("equal")

ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_color("#52738A")
    spine.set_linewidth(1.0)


image_main = ax.imshow(
    observed,
    origin="lower",
    extent=[
        -1.5, 1.5,
        -1.5, 1.5
    ],
    cmap="magma",
    vmin=0,
    vmax=0.55,
    interpolation="bicubic",
    zorder=1
)


# ------------------------------------------------------------
# IFU GRID
# ------------------------------------------------------------

grid_lines = []

for v in np.linspace(
    -1.5,
    1.5,
    16
):

    lv = ax.axvline(
        v,
        color="#A7D7EA",
        lw=0.38,
        alpha=0,
        zorder=5
    )

    lh = ax.axhline(
        v,
        color="#A7D7EA",
        lw=0.38,
        alpha=0,
        zorder=5
    )

    grid_lines.extend(
        [lv, lh]
    )


# ------------------------------------------------------------
# PSF RINGS
# ------------------------------------------------------------

psf_ring1 = Circle(
    (0, 0),
    0.25,
    fill=False,
    edgecolor="#D9F7FF",
    linewidth=1.3,
    alpha=0,
    zorder=10
)

psf_ring2 = Circle(
    (0, 0),
    0.48,
    fill=False,
    edgecolor="#7FDFFF",
    linewidth=0.8,
    alpha=0,
    zorder=10
)

ax.add_patch(psf_ring1)
ax.add_patch(psf_ring2)


# ------------------------------------------------------------
# TECHNICAL LABELS
# ------------------------------------------------------------

field_label = ax.text(
    -1.42, 1.05,
    'NIRSpec IFU  ~3" × 3"',
    ha="left",
    va="top",
    color="#DDF7FF",
    fontsize=11,
    fontweight="bold",
    zorder=20
)

spaxel_label = ax.text(
    -1.42, 0.88,
    'SPAXEL ≈ 0.1"',
    ha="left",
    va="top",
    color="#9FC9DC",
    fontsize=10,
    zorder=20
)

map_state = ax.text(
    0, -1.37,
    "QUASAR + HOST",
    ha="center",
    va="bottom",
    color="white",
    fontsize=13,
    fontweight="bold",
    zorder=20
)


# ------------------------------------------------------------
# PHASE TEXT
# ------------------------------------------------------------

phase_text = fig.text(
    0.32, 0.855,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.32, 0.818,
    "",
    ha="center",
    va="center",
    color="#B7D3E7",
    fontsize=11,
    zorder=50
)


# ============================================================
# SCENE 1 — RIGHT INFORMATION PANEL
# ============================================================

info = fig.add_axes(
    [0.635, 0.14, 0.325, 0.70],
    zorder=10
)

info.set_facecolor(BG)
info.axis("off")

info_title = info.text(
    0.5, 0.98,
    "HOW THE FAINT GAS IS REVEALED",
    ha="center",
    va="top",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold",
    transform=info.transAxes
)


# ============================================================
# H-BETA SPECTRUM
# ============================================================

spec_ax = fig.add_axes(
    [0.665, 0.57, 0.265, 0.18],
    zorder=20
)

spec_ax.set_facecolor(
    "#06101A"
)

vel = np.linspace(
    -2500,
    2500,
    600
)

broad_hbeta = np.exp(
    -0.5 *
    (vel / 900)**2
)

broad_hbeta += (
    0.12 *
    np.exp(
        -0.5 *
        (vel / 1800)**2
    )
)

spec_ax.plot(
    vel,
    broad_hbeta,
    color="#DDF8FF",
    lw=2.2
)

left_wing = spec_ax.axvspan(
    -1500,
    -500,
    color="#64D5FF",
    alpha=0
)

right_wing = spec_ax.axvspan(
    500,
    1500,
    color="#64D5FF",
    alpha=0
)

spec_ax.axvline(
    0,
    color="#638399",
    lw=0.8,
    alpha=0.5
)

spec_ax.set_xlim(
    -2500,
    2500
)

spec_ax.set_ylim(
    0,
    1.23
)

spec_ax.set_xlabel(
    "VELOCITY FROM Hβ  [km/s]",
    color="#AFC9D8",
    fontsize=9,
    fontweight="bold"
)

spec_ax.set_ylabel(
    "FLUX",
    color="#AFC9D8",
    fontsize=9,
    fontweight="bold"
)

spec_ax.tick_params(
    colors="#91ADBD",
    labelsize=8
)

for spine in spec_ax.spines.values():
    spine.set_color("#52738A")

spec_ax.grid(
    color="#52738A",
    alpha=0.14
)

spec_ax.set_title(
    "BROAD Hβ",
    color="#CDEEFF",
    fontsize=10,
    fontweight="bold"
)

wing_label = spec_ax.text(
    0.5, 0.82,
    "UNRESOLVED BLR → PSF MODEL",
    transform=spec_ax.transAxes,
    ha="center",
    color="#9BE9FF",
    fontsize=9,
    fontweight="bold",
    alpha=0
)


# ============================================================
# PROCEDURE STEPS
# ============================================================

step1 = info.text(
    0.06, 0.43,
    "1   OBSERVE",
    color="#DCEFFA",
    fontsize=12,
    fontweight="bold",
    transform=info.transAxes
)

step1b = info.text(
    0.14, 0.385,
    "Quasar + surrounding emission",
    color="#94B5C7",
    fontsize=10,
    transform=info.transAxes
)

step2 = info.text(
    0.06, 0.30,
    "2   MODEL PSF",
    color="#DCEFFA",
    fontsize=12,
    fontweight="bold",
    alpha=0.30,
    transform=info.transAxes
)

step2b = info.text(
    0.14, 0.255,
    "Use broad Hβ emission",
    color="#94B5C7",
    fontsize=10,
    alpha=0.30,
    transform=info.transAxes
)

step3 = info.text(
    0.06, 0.17,
    "3   SUBTRACT QUASAR",
    color="#DCEFFA",
    fontsize=12,
    fontweight="bold",
    alpha=0.30,
    transform=info.transAxes
)

step3b = info.text(
    0.14, 0.125,
    "Reveal spatially extended gas",
    color="#94B5C7",
    fontsize=10,
    alpha=0.30,
    transform=info.transAxes
)

step4 = info.text(
    0.06, 0.04,
    "4   MAP [O III]",
    color="#9CEBFF",
    fontsize=12,
    fontweight="bold",
    alpha=0.30,
    transform=info.transAxes
)


# ============================================================
# SCENE 2 — DARK BACKDROP
#
# Separate almost-black surface for the result screen.
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor(
    "#000207"
)

result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_alpha(0.0)
result_bg.set_visible(False)


# ============================================================
# SCENE 2 — RESULT MAPS
# ============================================================

flux_ax = fig.add_axes(
    [0.095, 0.34, 0.235, 0.40],
    zorder=80
)

vel_ax = fig.add_axes(
    [0.3825, 0.34, 0.235, 0.40],
    zorder=80
)

sig_ax = fig.add_axes(
    [0.670, 0.34, 0.235, 0.40],
    zorder=80
)

result_axes = [
    flux_ax,
    vel_ax,
    sig_ax
]

for a in result_axes:

    a.set_xticks([])
    a.set_yticks([])

    a.set_facecolor(
        "#03070D"
    )

    for spine in a.spines.values():
        spine.set_color(
            "#7895A8"
        )
        spine.set_linewidth(
            1.2
        )

    a.set_visible(False)


flux_im = flux_ax.imshow(
    nebula,
    origin="lower",
    cmap="magma",
    interpolation="bicubic"
)

vel_im = vel_ax.imshow(
    velocity,
    origin="lower",
    cmap="coolwarm",
    vmin=-300,
    vmax=300,
    interpolation="bicubic"
)

sig_im = sig_ax.imshow(
    dispersion,
    origin="lower",
    cmap="viridis",
    vmin=200,
    vmax=500,
    interpolation="bicubic"
)


flux_ax.set_title(
    "FLUX",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold",
    pad=9
)

vel_ax.set_title(
    "VELOCITY",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold",
    pad=9
)

sig_ax.set_title(
    "DISPERSION",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold",
    pad=9
)


# ============================================================
# SCENE 2 — TEXT
# ============================================================

result_header = fig.text(
    0.5, 0.825,
    "EXTENDED [O III] AROUND z ≈ 5–6 QUASARS",
    ha="center",
    va="center",
    color="#E5F7FF",
    fontsize=16,
    fontweight="bold",
    alpha=0,
    zorder=100
)

result_subheader = fig.text(
    0.5, 0.790,
    "Spatially resolved ionized gas after quasar subtraction",
    ha="center",
    va="center",
    color="#A8C9D9",
    fontsize=11,
    alpha=0,
    zorder=100
)

result_title = fig.text(
    0.5, 0.265,
    "6 / 27  EXTENDED [O III] NEBULAE",
    ha="center",
    va="center",
    color="white",
    fontsize=19,
    fontweight="bold",
    alpha=0,
    zorder=100
)

result_note = fig.text(
    0.5, 0.218,
    "≈22%  —  LOWER LIMIT",
    ha="center",
    va="center",
    color="#AEEAFF",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    zorder=100
)

result_detail = fig.text(
    0.5, 0.174,
    "|v| ≲ 300 km/s     •     σ ≈ 200–500 km/s",
    ha="center",
    va="center",
    color="#D0E2EB",
    fontsize=11,
    alpha=0,
    zorder=100
)

result_interpretation = fig.text(
    0.5, 0.132,
    "MERGERS  •  TURBULENT / CLUMPY ISM  •  COMPANIONS",
    ha="center",
    va="center",
    color="#9DB9C8",
    fontsize=10,
    fontweight="bold",
    alpha=0,
    zorder=100
)


# ============================================================
# INDEPENDENT TIME DIAL
#
# Decorative progress indicator only.
# NOT a physical timescale.
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(
    -1,
    1
)

clock_ax.set_ylim(
    -1,
    1
)

clock_ax.set_aspect("equal")
clock_ax.axis("off")


dial_center = (0, 0)
dial_radius = 0.82


dial_outer = Circle(
    dial_center,
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    dial_center,
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(
    dial_outer
)

clock_ax.add_patch(
    dial_inner
)


for i in range(12):

    angle_tick = (
        2 *
        np.pi *
        i / 12
    )

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = (
        dial_radius *
        0.83
    )

    x1 = (
        dial_center[0] +
        r1 *
        np.sin(angle_tick)
    )

    y1 = (
        dial_center[1] +
        r1 *
        np.cos(angle_tick)
    )

    x2 = (
        dial_center[0] +
        r2 *
        np.sin(angle_tick)
    )

    y2 = (
        dial_center[1] +
        r2 *
        np.cos(angle_tick)
    )

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )


hand_length = (
    dial_radius *
    0.56
)

dial_hand, = clock_ax.plot(
    [
        dial_center[0],
        dial_center[0]
    ],
    [
        dial_center[1],
        dial_center[1] +
        hand_length
    ],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    dial_center,
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(
    dial_pin
)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    # --------------------------------------------------------
    # REAL TIME
    # --------------------------------------------------------

    sec = frame / FPS

    # Scene-1 timing remains based on the original 18 s pace.
    # This prevents the first sequence from stretching when
    # the full animation is increased to 20 s.
    t1 = np.clip(
        sec / 18.0,
        0.0,
        1.0
    )

    # Full animation progress.
    # Used for the decorative clock only.
    t = frame / (
        NFRAMES - 1
    )


    # ========================================================
    # SCENE 1 — PHASES
    # ========================================================

    if t1 < 0.20:

        phase_text.set_text(
            "OBSERVE THE QUASAR"
        )

        phase_sub.set_text(
            "The bright unresolved nucleus dominates the IFU field"
        )

    elif t1 < 0.40:

        phase_text.set_text(
            "MODEL THE QUASAR PSF"
        )

        phase_sub.set_text(
            "Broad Hβ traces the unresolved nuclear source"
        )

    elif t1 < 0.59:

        phase_text.set_text(
            "SUBTRACT THE QUASAR"
        )

        phase_sub.set_text(
            "Remove the modeled point-source contribution"
        )

    else:

        phase_text.set_text(
            "EXTENDED [O III] APPEARS"
        )

        phase_sub.set_text(
            "Spatially resolved ionized gas remains"
        )


    # ========================================================
    # IFU GRID
    # ========================================================

    grid_progress = smoothstep(
        t1 / 0.12
    )

    for line in grid_lines:

        line.set_alpha(
            0.16 *
            grid_progress
        )


    # ========================================================
    # PSF MODEL
    # ========================================================

    psf_progress = smoothstep(
        (t1 - 0.18) / 0.16
    )

    subtraction = smoothstep(
        (t1 - 0.38) / 0.18
    )

    psf_alpha = (
        psf_progress *
        (1.0 - subtraction)
    )

    psf_ring1.set_alpha(
        0.85 *
        psf_alpha
    )

    psf_ring2.set_alpha(
        0.50 *
        psf_alpha
    )

    left_wing.set_alpha(
        0.18 *
        psf_progress
    )

    right_wing.set_alpha(
        0.18 *
        psf_progress
    )

    wing_label.set_alpha(
        psf_progress
    )

    step2.set_alpha(
        0.30 +
        0.70 *
        psf_progress
    )

    step2b.set_alpha(
        0.30 +
        0.70 *
        psf_progress
    )


    # ========================================================
    # QUASAR SUBTRACTION
    # ========================================================

    residual = (
        (1.0 - subtraction) *
        observed
        +
        subtraction *
        (
            0.02 +
            0.95 *
            nebula
        )
    )

    image_main.set_data(
        residual
    )

    image_main.set_clim(
        0,
        0.55 -
        0.18 *
        subtraction
    )

    step3.set_alpha(
        0.30 +
        0.70 *
        subtraction
    )

    step3b.set_alpha(
        0.30 +
        0.70 *
        subtraction
    )

    map_progress = smoothstep(
        (t1 - 0.54) / 0.10
    )

    step4.set_alpha(
        0.30 +
        0.70 *
        map_progress
    )


    # ========================================================
    # MAP STATE
    # ========================================================

    if subtraction < 0.15:

        map_state.set_text(
            "QUASAR + HOST"
        )

        map_state.set_color(
            "#FFFFFF"
        )

    elif subtraction < 0.85:

        map_state.set_text(
            "PSF SUBTRACTION"
        )

        map_state.set_color(
            "#A9E9FF"
        )

    else:

        map_state.set_text(
            "EXTENDED [O III]"
        )

        map_state.set_color(
            "#B6F7FF"
        )


    # ========================================================
    # SCENE TRANSITION
    #
    # 13.2 s: begins
    # 14.0 s: final scene fully visible
    # 14–20 s: six-second hold
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = (
        1.0 -
        scene2
    )


    # --------------------------------------------------------
    # Fade first scene
    # --------------------------------------------------------

    ax.set_alpha(
        scene1_alpha
    )

    info.set_alpha(
        scene1_alpha
    )

    spec_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )


    # ========================================================
    # SECOND SCENE — DARK BACKGROUND
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(
            True
        )

        result_bg.set_alpha(
            0.97 *
            scene2
        )

    else:

        result_bg.set_visible(
            False
        )


    # ========================================================
    # SECOND SCENE — MAPS
    # ========================================================

    if scene2 > 0.005:

        for a in result_axes:

            a.set_visible(
                True
            )

            a.set_alpha(
                scene2
            )

    else:

        for a in result_axes:

            a.set_visible(
                False
            )


    # ========================================================
    # SECOND SCENE — TEXT
    # ========================================================

    result_header.set_alpha(
        scene2
    )

    result_subheader.set_alpha(
        scene2
    )

    result_title.set_alpha(
        scene2
    )

    result_note.set_alpha(
        scene2
    )

    result_detail.set_alpha(
        scene2
    )

    result_interpretation.set_alpha(
        scene2
    )


    # ========================================================
    # TIME DIAL
    #
    # One clockwise revolution over full 20 s.
    # ========================================================

    dial_angle = (
        2.0 *
        np.pi *
        t
    )

    hand_x = (
        dial_center[0]
        +
        hand_length *
        np.sin(
            dial_angle
        )
    )

    hand_y = (
        dial_center[1]
        +
        hand_length *
        np.cos(
            dial_angle
        )
    )

    dial_hand.set_data(
        [
            dial_center[0],
            hand_x
        ],
        [
            dial_center[1],
            hand_y
        ]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)


print(
    f"Saved: {OUTFILE}"
)

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# Как JWST обнаруживает газ вокруг далёких квазаров

Яркое излучение квазара может почти полностью скрывать гораздо более слабое свечение окружающего его газа. Чтобы отделить одно от другого, астрономы используют интегральный спектрограф JWST/NIRSpec IFU, который получает спектр для каждого небольшого участка изображения.

Авторы исследовали 27 ярких квазаров с красным смещением z ≈ 5–6. Для построения модели изображения центрального квазара они использовали широкую линию водорода Hβ, возникающую в компактной области возле сверхмассивной чёрной дыры. Поскольку эта область пространственно не разрешается, её излучение позволяет восстановить профиль точечного источника — PSF.

После вычитания модели квазара из данных становится видно слабое протяжённое излучение дважды ионизованного кислорода [O III]. Такие структуры были обнаружены у 6 из 27 квазаров — примерно у 22% выборки. Авторы считают эту долю нижней границей: наблюдения были сравнительно короткими, поэтому более слабые туманности могли остаться незамеченными.

Обнаруженный газ имеет сложную асимметричную структуру. Его скорости преимущественно находятся в пределах примерно ±300 км/с относительно квазара, а дисперсия скоростей составляет около 200–500 км/с. Наблюдаемая картина может быть связана со слияниями галактик, турбулентной неоднородной межзвёздной средой и галактиками-компаньонами.

Так JWST позволяет буквально убрать ослепительный свет квазара и увидеть среду галактики, в которой уже в течение первого примерно миллиарда лет существования Вселенной росли сверхмассивные чёрные дыры.

Анимация схематическая. Распределение газа и карты IFU иллюстрируют метод исследования и не воспроизводят конкретный объект из выборки.

# How JWST Reveals Gas Around Distant Quasars

The intense light of a quasar can almost completely overwhelm the much fainter emission from surrounding gas. To separate the two, astronomers use the JWST/NIRSpec integral field unit, which records a spectrum for each small spatial element across the observed field.

The researchers studied 27 luminous quasars at redshifts of z ≈ 5–6. To model the central quasar, they used the broad hydrogen Hβ emission line produced in the compact region close to the supermassive black hole. Because this region is spatially unresolved, its emission can be used to reconstruct the point-spread function, or PSF, of the quasar.

After subtracting the modeled quasar contribution, faint spatially extended emission from doubly ionized oxygen, [O III], becomes visible. Such structures were detected around 6 of the 27 quasars — approximately 22% of the sample. The authors regard this fraction as a lower limit because the observations were relatively shallow and fainter nebulae may remain undetected.

The detected gas has complex, asymmetric structures. Its velocities are mostly within about ±300 km/s of the quasars, with velocity dispersions of approximately 200–500 km/s. These structures may be associated with galaxy mergers, a turbulent and clumpy interstellar medium, and companion galaxies.

In this way, JWST can effectively remove the dazzling light of a quasar and reveal the galactic environment in which supermassive black holes were already growing within roughly the first billion years of cosmic history.

The animation is schematic. The gas distribution and IFU maps illustrate the observational technique and do not reconstruct any individual object in the sample.

---

# Part 3. Supermassive Black Holes in the Early Universe

In [ ]:
# ============================================================
# BILLION-SOLAR-MASS BLACK HOLES IN THE EARLY UNIVERSE
# Inspired by Liu et al. (2026), ApJ 1009:176
#
# One self-contained Jupyter cell
# Requires: numpy, matplotlib, ffmpeg
#
# STORY
#   0–6 s     Quasar + broad H-beta line
#   6–13.2 s  H-beta width + continuum -> virial BH mass
#   13.2–14 s Transition
#   14–20 s   Final result screen
#
# IMPORTANT
# This animation illustrates the measurement concept.
# It does NOT show the physical growth history of a black hole.
# Spectral profiles and geometry are schematic.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Circle, Ellipse, Polygon
from IPython.display import Video, display


# ============================================================
# OUTPUT
# ============================================================

OUTPUT = "mp4"       # "webm" or "mp4"

FPS = 30
DURATION = 20
NFRAMES = FPS * DURATION
DPI = 120

if OUTPUT == "webm":
    OUTFILE = "ae99e0_early_universe_billion_solar_mass_bh.webm"
else:
    OUTFILE = "ae99e0_early_universe_billion_solar_mass_bh.mp4"


# ============================================================
# HELPERS
# ============================================================

def clamp(x, a=0.0, b=1.0):
    return np.minimum(np.maximum(x, a), b)


def smoothstep(x):
    x = clamp(x)
    return x * x * (3.0 - 2.0 * x)


BG = "#02050b"

rng = np.random.default_rng(7)


# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(
    figsize=(12.8, 7.2),
    facecolor=BG
)


# ============================================================
# GLOBAL TITLE
# ============================================================

main_title = fig.text(
    0.5, 0.955,
    "MEASURING BLACK HOLES IN z ≈ 5–6 QUASARS",
    ha="center",
    va="center",
    color="white",
    fontsize=20,
    fontweight="bold",
    zorder=200
)

main_subtitle = fig.text(
    0.5, 0.918,
    "Broad Hβ emission reveals the mass of the central black hole",
    ha="center",
    va="center",
    color="#B8D4E8",
    fontsize=12,
    zorder=200
)


# ============================================================
# LEFT PANEL — QUASAR
# ============================================================

ax = fig.add_axes(
    [0.045, 0.14, 0.48, 0.70],
    zorder=10
)

ax.set_facecolor(BG)

ax.set_xlim(
    -1.35,
    1.35
)

ax.set_ylim(
    -1.05,
    1.05
)

ax.set_aspect("equal")
ax.axis("off")


# ------------------------------------------------------------
# HOST GALAXY
# ------------------------------------------------------------

N_STARS = 650

theta = rng.uniform(
    0,
    2 * np.pi,
    N_STARS
)

radius = rng.gamma(
    shape=1.6,
    scale=0.31,
    size=N_STARS
)

radius = np.clip(
    radius,
    0,
    1.15
)

sx = (
    radius *
    np.cos(theta)
)

sy = (
    0.46 *
    radius *
    np.sin(theta)
)

# slight spiral distortion
sx += (
    0.08 *
    np.sin(
        2.5 * theta +
        3.0 * radius
    )
)

star_alpha = (
    0.06 +
    0.34 *
    np.exp(
        -1.7 * radius
    )
)

star_colors = np.ones(
    (N_STARS, 4)
)

star_colors[:, 0] = 0.68
star_colors[:, 1] = 0.78
star_colors[:, 2] = 0.88
star_colors[:, 3] = star_alpha

host_scatter = ax.scatter(
    sx,
    sy,
    s=rng.uniform(
        1.0,
        5.0,
        N_STARS
    ),
    c=star_colors,
    linewidths=0,
    zorder=1
)


# ------------------------------------------------------------
# QUASAR GLOW
# ------------------------------------------------------------

glow1 = Circle(
    (0, 0),
    0.42,
    facecolor="#4BA9D8",
    edgecolor="none",
    alpha=0.035,
    zorder=3
)

glow2 = Circle(
    (0, 0),
    0.25,
    facecolor="#86DDFF",
    edgecolor="none",
    alpha=0.07,
    zorder=4
)

glow3 = Circle(
    (0, 0),
    0.12,
    facecolor="white",
    edgecolor="none",
    alpha=0.15,
    zorder=5
)

ax.add_patch(glow1)
ax.add_patch(glow2)
ax.add_patch(glow3)


# ------------------------------------------------------------
# ACCRETION DISK
# ------------------------------------------------------------

disk_outer = Ellipse(
    (0, 0),
    0.72,
    0.18,
    angle=0,
    facecolor="#57BFE9",
    edgecolor="#B8EEFF",
    linewidth=1.0,
    alpha=0.36,
    zorder=8
)

disk_mid = Ellipse(
    (0, 0),
    0.49,
    0.105,
    facecolor="#A5E9FF",
    edgecolor="none",
    alpha=0.48,
    zorder=9
)

disk_inner = Ellipse(
    (0, 0),
    0.25,
    0.052,
    facecolor="white",
    edgecolor="none",
    alpha=0.76,
    zorder=10
)

ax.add_patch(disk_outer)
ax.add_patch(disk_mid)
ax.add_patch(disk_inner)


# ------------------------------------------------------------
# BLACK HOLE
# ------------------------------------------------------------

black_hole = Circle(
    (0, 0),
    0.065,
    facecolor="#000000",
    edgecolor="#DDF8FF",
    linewidth=0.8,
    zorder=14
)

ax.add_patch(black_hole)


# ------------------------------------------------------------
# BROAD-LINE REGION
# ------------------------------------------------------------

N_BLR = 95

blr_theta = rng.uniform(
    0,
    2 * np.pi,
    N_BLR
)

blr_r = rng.uniform(
    0.17,
    0.42,
    N_BLR
)

blr_x = (
    blr_r *
    np.cos(blr_theta)
)

blr_y = (
    0.35 *
    blr_r *
    np.sin(blr_theta)
)

blr_scatter = ax.scatter(
    blr_x,
    blr_y,
    s=rng.uniform(
        5,
        15,
        N_BLR
    ),
    color="#83E4FF",
    alpha=0,
    linewidths=0,
    zorder=12
)


# ------------------------------------------------------------
# LABELS
# ------------------------------------------------------------

bh_label = ax.text(
    0,
    -0.62,
    "ACTIVE GALACTIC NUCLEUS",
    ha="center",
    va="center",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold",
    zorder=30
)

blr_label = ax.text(
    0,
    -0.76,
    "BROAD-LINE REGION",
    ha="center",
    va="center",
    color="#94C7DC",
    fontsize=10,
    alpha=0,
    zorder=30
)


# ============================================================
# RIGHT PANEL — H-BETA SPECTRUM
# ============================================================

spec_ax = fig.add_axes(
    [0.585, 0.48, 0.36, 0.29],
    zorder=10
)

spec_ax.set_facecolor(
    "#06101A"
)

velocity_axis = np.linspace(
    -8000,
    8000,
    900
)

# schematic broad H-beta
sigma_hb = 2050.0

hbeta = np.exp(
    -0.5 *
    (
        velocity_axis /
        sigma_hb
    ) ** 2
)

hbeta += (
    0.08 *
    np.exp(
        -0.5 *
        (
            velocity_axis /
            4300
        ) ** 2
    )
)

spec_line, = spec_ax.plot(
    velocity_axis,
    hbeta,
    color="#E6F9FF",
    lw=2.5
)

spec_ax.set_xlim(
    -8000,
    8000
)

spec_ax.set_ylim(
    0,
    1.18
)

spec_ax.set_xlabel(
    "VELOCITY FROM Hβ  [km/s]",
    color="#B5CDE0",
    fontsize=10,
    fontweight="bold"
)

spec_ax.set_ylabel(
    "FLUX",
    color="#B5CDE0",
    fontsize=10,
    fontweight="bold"
)

spec_ax.tick_params(
    colors="#AFC7D9",
    labelsize=9
)

for spine in spec_ax.spines.values():
    spine.set_color("#52738A")
    spine.set_linewidth(1.0)

spec_ax.grid(
    color="#52738A",
    alpha=0.16
)

spec_ax.set_title(
    "BROAD Hβ",
    color="#DDF5FF",
    fontsize=12,
    fontweight="bold"
)


# ------------------------------------------------------------
# FWHM INDICATOR
# ------------------------------------------------------------

half_level = 0.5

fwhm_left = -4825
fwhm_right = 4825

fwhm_line, = spec_ax.plot(
    [fwhm_left, fwhm_right],
    [half_level, half_level],
    color="#72DFFF",
    lw=2.0,
    alpha=0
)

fwhm_cap1, = spec_ax.plot(
    [fwhm_left, fwhm_left],
    [0.44, 0.56],
    color="#72DFFF",
    lw=2,
    alpha=0
)

fwhm_cap2, = spec_ax.plot(
    [fwhm_right, fwhm_right],
    [0.44, 0.56],
    color="#72DFFF",
    lw=2,
    alpha=0
)

fwhm_text = spec_ax.text(
    0,
    0.59,
    "FWHM",
    ha="center",
    va="bottom",
    color="#8EE8FF",
    fontsize=10,
    fontweight="bold",
    alpha=0
)


# ============================================================
# RIGHT PANEL — MEASUREMENT LOGIC
# ============================================================

measure_ax = fig.add_axes(
    [0.585, 0.14, 0.36, 0.25],
    zorder=10
)

measure_ax.set_facecolor(BG)
measure_ax.axis("off")


measure_1 = measure_ax.text(
    0.5, 0.86,
    "Hβ WIDTH",
    ha="center",
    va="center",
    color="#DCEFFA",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    transform=measure_ax.transAxes
)

measure_plus = measure_ax.text(
    0.5, 0.66,
    "+",
    ha="center",
    va="center",
    color="#7698AA",
    fontsize=17,
    fontweight="bold",
    alpha=0,
    transform=measure_ax.transAxes
)

measure_2 = measure_ax.text(
    0.5, 0.49,
    "CONTINUUM LUMINOSITY",
    ha="center",
    va="center",
    color="#DCEFFA",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    transform=measure_ax.transAxes
)

measure_arrow = measure_ax.text(
    0.5, 0.29,
    "↓",
    ha="center",
    va="center",
    color="#7EDFFF",
    fontsize=19,
    fontweight="bold",
    alpha=0,
    transform=measure_ax.transAxes
)

measure_result = measure_ax.text(
    0.5, 0.09,
    "VIRIAL BLACK-HOLE MASS",
    ha="center",
    va="center",
    color="#A9EEFF",
    fontsize=13,
    fontweight="bold",
    alpha=0,
    transform=measure_ax.transAxes
)


# ============================================================
# PHASE TEXT
# ============================================================

phase_text = fig.text(
    0.285, 0.855,
    "",
    ha="center",
    va="center",
    color="white",
    fontsize=16,
    fontweight="bold",
    zorder=50
)

phase_sub = fig.text(
    0.285, 0.818,
    "",
    ha="center",
    va="center",
    color="#B7D3E7",
    fontsize=11,
    zorder=50
)


# ============================================================
# MASS RANGE PANEL
# ============================================================

mass_ax = fig.add_axes(
    [0.12, 0.065, 0.76, 0.085],
    zorder=20
)

mass_ax.set_facecolor(BG)

mass_ax.set_xlim(
    8.0,
    10.0
)

mass_ax.set_ylim(
    0,
    1
)

mass_ax.set_yticks([])

mass_ax.set_xticks(
    [8.0, 8.5, 9.0, 9.5, 10.0]
)

mass_ax.set_xticklabels(
    [
        r"$10^{8.0}$",
        r"$10^{8.5}$",
        r"$10^{9.0}$",
        r"$10^{9.5}$",
        r"$10^{10}$"
    ]
)

mass_ax.tick_params(
    axis="x",
    colors="#AFC7D9",
    labelsize=9
)

for spine in mass_ax.spines.values():
    spine.set_visible(False)

mass_ax.set_xlabel(
    "BLACK-HOLE MASS  [M☉]",
    color="#AFC7D9",
    fontsize=10,
    fontweight="bold"
)

mass_ax.plot(
    [8.0, 10.0],
    [0.50, 0.50],
    color="#52738A",
    lw=2
)

range_line, = mass_ax.plot(
    [8.6, 8.6],
    [0.50, 0.50],
    color="#7EDFFF",
    lw=7,
    solid_capstyle="round",
    alpha=0
)

mass_marker, = mass_ax.plot(
    [8.6],
    [0.50],
    marker="o",
    markersize=9,
    color="#E8FAFF",
    alpha=0
)

range_text = mass_ax.text(
    9.15,
    0.78,
    "Q-IFU SAMPLE",
    ha="center",
    va="center",
    color="#BDEFFF",
    fontsize=9,
    fontweight="bold",
    alpha=0
)


# ============================================================
# FINAL SCENE — DARK BACKDROP
# ============================================================

result_bg = fig.add_axes(
    [0.025, 0.075, 0.95, 0.80],
    zorder=70
)

result_bg.set_facecolor(
    "#000207"
)

result_bg.set_xticks([])
result_bg.set_yticks([])

for spine in result_bg.spines.values():
    spine.set_visible(False)

result_bg.set_visible(False)
result_bg.set_alpha(0)


# ============================================================
# FINAL SCENE — CENTRAL BLACK HOLE
# ============================================================

final_ax = fig.add_axes(
    [0.09, 0.30, 0.34, 0.46],
    zorder=80
)

final_ax.set_facecolor(
    "#000207"
)

final_ax.set_xlim(
    -1,
    1
)

final_ax.set_ylim(
    -1,
    1
)

final_ax.set_aspect("equal")
final_ax.axis("off")
final_ax.set_visible(False)


# outer glow
for radius, alpha in [
    (0.78, 0.025),
    (0.61, 0.040),
    (0.46, 0.065),
    (0.32, 0.10)
]:

    final_ax.add_patch(
        Circle(
            (0, 0),
            radius,
            facecolor="#55BFEA",
            edgecolor="none",
            alpha=alpha
        )
    )


# accretion disk
final_ax.add_patch(
    Ellipse(
        (0, 0),
        1.55,
        0.31,
        facecolor="#4EBBE9",
        edgecolor="#BCEFFF",
        linewidth=1.1,
        alpha=0.36,
        zorder=5
    )
)

final_ax.add_patch(
    Ellipse(
        (0, 0),
        1.10,
        0.19,
        facecolor="#9BE7FF",
        edgecolor="none",
        alpha=0.53,
        zorder=6
    )
)

final_ax.add_patch(
    Ellipse(
        (0, 0),
        0.64,
        0.095,
        facecolor="white",
        edgecolor="none",
        alpha=0.74,
        zorder=7
    )
)

final_ax.add_patch(
    Circle(
        (0, 0),
        0.19,
        facecolor="#000000",
        edgecolor="#DDF8FF",
        linewidth=1.1,
        zorder=10
    )
)


# ============================================================
# FINAL SCENE — MASS SCALE
# ============================================================

final_mass_ax = fig.add_axes(
    [0.49, 0.47, 0.43, 0.18],
    zorder=85
)

final_mass_ax.set_facecolor(
    "#000207"
)

final_mass_ax.set_xlim(
    8.0,
    10.0
)

final_mass_ax.set_ylim(
    0,
    1
)

final_mass_ax.set_yticks([])

final_mass_ax.set_xticks(
    [8, 9, 10]
)

final_mass_ax.set_xticklabels(
    [
        r"$10^8$",
        r"$10^9$",
        r"$10^{10}$"
    ]
)

final_mass_ax.tick_params(
    axis="x",
    colors="#C3DCE8",
    labelsize=11
)

for spine in final_mass_ax.spines.values():
    spine.set_visible(False)

final_mass_ax.set_xlabel(
    "BLACK-HOLE MASS  [M☉]",
    color="#AFC9D8",
    fontsize=10,
    fontweight="bold"
)

final_mass_ax.plot(
    [8, 10],
    [0.52, 0.52],
    color="#526F80",
    lw=3
)

final_mass_ax.plot(
    [8.6, 9.7],
    [0.52, 0.52],
    color="#79DFFF",
    lw=10,
    solid_capstyle="round"
)

final_mass_ax.plot(
    [8.6, 9.7],
    [0.52, 0.52],
    marker="|",
    markersize=16,
    markeredgewidth=2,
    color="#E5FAFF",
    linestyle="None"
)

final_mass_ax.text(
    9.15,
    0.75,
    "Q-IFU RANGE",
    ha="center",
    va="center",
    color="#C8F2FF",
    fontsize=10,
    fontweight="bold"
)

final_mass_ax.set_visible(False)


# ============================================================
# FINAL SCENE — TEXT
# ============================================================

final_header = fig.text(
    0.70, 0.755,
    "MASSIVE BLACK HOLES\nIN THE EARLY UNIVERSE",
    ha="center",
    va="center",
    color="white",
    fontsize=18,
    fontweight="bold",
    linespacing=1.25,
    alpha=0,
    zorder=100
)

final_epoch = fig.text(
    0.70, 0.685,
    "already within the first ~1 billion years",
    ha="center",
    va="center",
    color="#A9D9EA",
    fontsize=11,
    alpha=0,
    zorder=100
)

final_sample = fig.text(
    0.70, 0.385,
    "27 LUMINOUS QUASARS  •  z ≈ 5–6",
    ha="center",
    va="center",
    color="#D8EDF6",
    fontsize=12,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_mass_text = fig.text(
    0.70, 0.330,
    "log(MBH / M☉) ≈ 8.6–9.7",
    ha="center",
    va="center",
    color="#AEEAFF",
    fontsize=14,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_edd = fig.text(
    0.70, 0.280,
    "λEdd ≈ 0.1–2.6",
    ha="center",
    va="center",
    color="#BCD5E1",
    fontsize=12,
    fontweight="bold",
    alpha=0,
    zorder=100
)

final_method = fig.text(
    0.70, 0.220,
    "Hβ-BASED VIRIAL MASS ESTIMATES",
    ha="center",
    va="center",
    color="#8EADBC",
    fontsize=10,
    alpha=0,
    zorder=100
)


# ============================================================
# INDEPENDENT TIME DIAL
# ============================================================

clock_ax = fig.add_axes(
    [0.030, 0.035, 0.080, 0.14],
    facecolor="none",
    zorder=300
)

clock_ax.set_xlim(-1, 1)
clock_ax.set_ylim(-1, 1)
clock_ax.set_aspect("equal")
clock_ax.axis("off")

dial_center = (0, 0)
dial_radius = 0.82

dial_outer = Circle(
    dial_center,
    dial_radius,
    facecolor="#07111A",
    edgecolor="#91DDF7",
    linewidth=1.3,
    alpha=0.80,
    zorder=30
)

dial_inner = Circle(
    dial_center,
    dial_radius * 0.82,
    facecolor="none",
    edgecolor="#52788C",
    linewidth=0.7,
    alpha=0.65,
    zorder=31
)

clock_ax.add_patch(
    dial_outer
)

clock_ax.add_patch(
    dial_inner
)

for i in range(12):

    a = (
        2 *
        np.pi *
        i / 12
    )

    r1 = (
        dial_radius *
        (0.67 if i % 3 else 0.59)
    )

    r2 = (
        dial_radius *
        0.83
    )

    x1 = (
        r1 *
        np.sin(a)
    )

    y1 = (
        r1 *
        np.cos(a)
    )

    x2 = (
        r2 *
        np.sin(a)
    )

    y2 = (
        r2 *
        np.cos(a)
    )

    clock_ax.plot(
        [x1, x2],
        [y1, y2],
        color="#A7D8EA",
        lw=1.1 if i % 3 else 1.6,
        alpha=0.72,
        zorder=32
    )

hand_length = (
    dial_radius *
    0.56
)

dial_hand, = clock_ax.plot(
    [0, 0],
    [0, hand_length],
    color="#E8FAFF",
    lw=1.8,
    solid_capstyle="round",
    zorder=34
)

dial_pin = Circle(
    (0, 0),
    dial_radius * 0.065,
    facecolor="#E8FAFF",
    edgecolor="none",
    zorder=35
)

clock_ax.add_patch(
    dial_pin
)


# ============================================================
# UPDATE
# ============================================================

def update(frame):

    sec = frame / FPS

    t = frame / (
        NFRAMES - 1
    )


    # ========================================================
    # PHASE TEXT
    # ========================================================

    if sec < 3.0:

        phase_text.set_text(
            "LUMINOUS QUASAR"
        )

        phase_sub.set_text(
            "A rapidly accreting supermassive black hole"
        )

    elif sec < 6.0:

        phase_text.set_text(
            "BROAD Hβ EMISSION"
        )

        phase_sub.set_text(
            "Fast gas orbits in the unresolved broad-line region"
        )

    elif sec < 9.5:

        phase_text.set_text(
            "MEASURE THE LINE WIDTH"
        )

        phase_sub.set_text(
            "Hβ provides a characteristic velocity scale"
        )

    else:

        phase_text.set_text(
            "ESTIMATE THE BLACK-HOLE MASS"
        )

        phase_sub.set_text(
            "Line width + continuum luminosity → virial mass"
        )


    # ========================================================
    # QUASAR PULSE
    # ========================================================

    pulse = (
        0.5 +
        0.5 *
        np.sin(
            2 *
            np.pi *
            sec / 2.7
        )
    )

    glow2.set_alpha(
        0.055 +
        0.035 * pulse
    )

    glow3.set_alpha(
        0.13 +
        0.06 * pulse
    )


    # ========================================================
    # BROAD-LINE REGION APPEARS
    # ========================================================

    blr_progress = smoothstep(
        (sec - 2.2) / 1.5
    )

    blr_scatter.set_alpha(
        0.58 *
        blr_progress
    )

    blr_label.set_alpha(
        blr_progress
    )


    # ========================================================
    # FWHM MEASUREMENT
    # ========================================================

    fwhm_progress = smoothstep(
        (sec - 5.0) / 1.3
    )

    fwhm_line.set_alpha(
        fwhm_progress
    )

    fwhm_cap1.set_alpha(
        fwhm_progress
    )

    fwhm_cap2.set_alpha(
        fwhm_progress
    )

    fwhm_text.set_alpha(
        fwhm_progress
    )


    # ========================================================
    # VIRIAL-MASS LOGIC
    # ========================================================

    measure_progress = smoothstep(
        (sec - 6.4) / 1.4
    )

    measure_1.set_alpha(
        measure_progress
    )

    measure_plus.set_alpha(
        smoothstep(
            (sec - 7.0) / 0.7
        )
    )

    measure_2.set_alpha(
        smoothstep(
            (sec - 7.4) / 0.8
        )
    )

    measure_arrow.set_alpha(
        smoothstep(
            (sec - 8.1) / 0.7
        )
    )

    measure_result.set_alpha(
        smoothstep(
            (sec - 8.6) / 0.9
        )
    )


    # ========================================================
    # SAMPLE MASS RANGE
    #
    # This is NOT black-hole growth.
    # It progressively reveals the measured range.
    # ========================================================

    mass_progress = smoothstep(
        (sec - 8.6) / 3.2
    )

    current_mass = (
        8.6 +
        (9.7 - 8.6) *
        mass_progress
    )

    range_line.set_data(
        [8.6, current_mass],
        [0.50, 0.50]
    )

    range_line.set_alpha(
        mass_progress
    )

    mass_marker.set_data(
        [current_mass],
        [0.50]
    )

    mass_marker.set_alpha(
        mass_progress
    )

    range_text.set_alpha(
        mass_progress
    )


    # ========================================================
    # FINAL SCENE TRANSITION
    #
    # 13.2 s -> transition begins
    # 14.0 s -> final scene complete
    # 14–20 s -> six-second hold
    # ========================================================

    scene2 = smoothstep(
        (sec - 13.2) / 0.8
    )

    scene1_alpha = (
        1.0 -
        scene2
    )


    ax.set_alpha(
        scene1_alpha
    )

    spec_ax.set_alpha(
        scene1_alpha
    )

    measure_ax.set_alpha(
        scene1_alpha
    )

    mass_ax.set_alpha(
        scene1_alpha
    )

    phase_text.set_alpha(
        scene1_alpha
    )

    phase_sub.set_alpha(
        scene1_alpha
    )

    main_subtitle.set_alpha(
        scene1_alpha
    )


    # ========================================================
    # DARK FINAL BACKGROUND
    # ========================================================

    if scene2 > 0.005:

        result_bg.set_visible(
            True
        )

        result_bg.set_alpha(
            0.97 *
            scene2
        )

        final_ax.set_visible(
            True
        )

        final_ax.set_alpha(
            scene2
        )

        final_mass_ax.set_visible(
            True
        )

        final_mass_ax.set_alpha(
            scene2
        )

    else:

        result_bg.set_visible(
            False
        )

        final_ax.set_visible(
            False
        )

        final_mass_ax.set_visible(
            False
        )


    # ========================================================
    # FINAL TEXT
    # ========================================================

    final_header.set_alpha(
        scene2
    )

    final_epoch.set_alpha(
        scene2
    )

    final_sample.set_alpha(
        scene2
    )

    final_mass_text.set_alpha(
        scene2
    )

    final_edd.set_alpha(
        scene2
    )

    final_method.set_alpha(
        scene2
    )


    # ========================================================
    # TIME DIAL
    # ========================================================

    dial_angle = (
        2.0 *
        np.pi *
        t
    )

    hand_x = (
        hand_length *
        np.sin(
            dial_angle
        )
    )

    hand_y = (
        hand_length *
        np.cos(
            dial_angle
        )
    )

    dial_hand.set_data(
        [0, hand_x],
        [0, hand_y]
    )


    return []


# ============================================================
# ANIMATION
# ============================================================

anim = FuncAnimation(
    fig,
    update,
    frames=NFRAMES,
    interval=1000 / FPS,
    blit=False
)


# ============================================================
# EXPORT
# ============================================================

if OUTPUT == "webm":

    writer = FFMpegWriter(
        fps=FPS,
        codec="libvpx-vp9",
        bitrate=2400,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-row-mt", "1",
            "-deadline", "good",
            "-cpu-used", "2"
        ]
    )

else:

    writer = FFMpegWriter(
        fps=FPS,
        codec="libx264",
        bitrate=4000,
        extra_args=[
            "-pix_fmt", "yuv420p",
            "-preset", "medium",
            "-movflags", "+faststart"
        ]
    )


anim.save(
    OUTFILE,
    writer=writer,
    dpi=DPI
)

plt.close(fig)

print(
    f"Saved: {OUTFILE}"
)

display(
    Video(
        OUTFILE,
        embed=True,
        html_attributes="controls loop"
    )
)

# Сверхмассивные чёрные дыры в ранней Вселенной

С помощью JWST/NIRSpec астрономы исследовали 27 ярких квазаров с красным смещением z ≈ 5–6 — объектов, существовавших в течение первого примерно миллиарда лет истории Вселенной.

Широкая эмиссионная линия водорода Hβ позволяет оценить скорость газа, движущегося в компактной области вокруг сверхмассивной чёрной дыры. Совместив ширину этой линии со светимостью квазара, исследователи получают вириальную оценку массы центральной чёрной дыры.

Для квазаров выборки получены массы примерно от 10^8.6 до 10^9.7 масс Солнца. Эддингтоновские отношения составляют приблизительно 0.1–2.6, что показывает высокую активность аккреции у этих объектов.

Таким образом, уже в ранней Вселенной существовали активно растущие чёрные дыры с массами порядка миллиарда солнечных. Наблюдения не показывают непосредственно, как эти объекты выросли до таких масс; они измеряют их свойства в наблюдаемую эпоху.

Анимация схематическая. Профиль Hβ, строение области широких линий и изображение аккреционного диска служат для объяснения метода измерения и не воспроизводят конкретный квазар.

# Supermassive Black Holes in the Early Universe

Using JWST/NIRSpec, astronomers studied 27 luminous quasars at redshifts of z ≈ 5–6, observing objects that already existed within roughly the first billion years of cosmic history.

The broad hydrogen Hβ emission line provides a measure of the characteristic velocity of gas moving in the compact region surrounding the supermassive black hole. Combining the width of this line with the quasar continuum luminosity allows astronomers to estimate the black hole’s virial mass.

The quasars in the sample host black holes with estimated masses of approximately 10^8.6–10^9.7 solar masses. Their Eddington ratios span roughly 0.1–2.6, indicating highly active accretion in these systems.

The observations therefore show that actively accreting black holes with masses around a billion Suns were already present in the early Universe. They do not directly reveal how individual black holes grew to these enormous masses; they measure their properties at the observed epoch.

The animation is schematic. The Hβ profile, broad-line region, and accretion-disk visualization illustrate the measurement concept and do not reproduce any individual quasar in the sample.